# Pre- vs. post-SHINE cohort comparison: similarity structure

Do the pre-SHINE and post-SHINE cohorts induce the *same perceptual similarity structure* -- i.e. does SHINE preprocessing change the geometry of subjects' similarity judgements?<br>
Compares the two cohorts' behavioral RDMs directly and through weighted-SMACOF MDS embeddings, for **retained subjects only** (production cohort, `status == "full data"`, `task_version >= 4.0`).<br>
Companion notebooks:
- `1_demographics_and_behavior.ipynb` - do they behave the same?
- `2_data_quality.ipynb` - is the data equally good?.

Figure helpers live in `analysis/behavioral/cohort_structure.py`.

Each cohort's full 725x725 behavioral RDM is built via `analysis.build_rdms.behavior.spam_ratings.build_behavioral_rdm()`
(`aggregation="simple"`, the project's validated default per `aggregation_comparison.ipynb`) -- a condensed
distance vector with NaN for the ~35-37% of pairs no retained subject in that cohort ever judged.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/behavioral/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from functools import partial

import pandas as pd

from analysis.utils.stats import masked_rank_correlation
from analysis.utils.equivalence import print_equivalence_line
from analysis.utils.plotting import fig_equivalence_forest
from analysis.build_rdms.behavior.spam_ratings import build_behavioral_rdm
from analysis.build_rdms.behavior.mds import embedding_eigenspectrum, run_mds_sweep
from SpAM_Simulations.core.helpers import procrustes_m2
from analysis.behavioral.demographics_and_behavior import COHORTS, COHORT_LABEL, load_retained_cohorts
from analysis.behavioral.cohort_structure import (
    MDS_CACHE_DIR, PERMUTATION_CACHE_DIR, fig_stress_scree, fig_embedding_spectrum, fig_cross_cohort_agreement,
    parse_subject_trials, rdm_from_subject_trials, permutation_test_cohort_agreement, fig_permutation_null,
    equivalence_test_cohort_agreement,
)

## 0. Load data

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
rdm = {c: build_behavioral_rdm(f"{c}_shine", cache_dir=cache_dir, aggregation="simple") for c in COHORTS}

[behavioral] variant='pre_shine': 79 participants.


[behavioral] Built 1738 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 64.57% covered (92977 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.95, sd=1.13.
[behavioral] WARNING: 92977 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_pre_simple.npy  (length 262450)


[behavioral] variant='post_shine': 76 participants.


[behavioral] Built 1672 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 63.09% covered (96882 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.92, sd=1.11.
[behavioral] WARNING: 96882 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_post_simple.npy  (length 262450)


## 1. Structural comparison via MDS

### 1.1 Weighted-SMACOF embeddings (R `smacof` backend)

Metric SMACOF via R's `smacof::mds()` (`SpAM_Simulations/core/multi_dimensional_scaling.py`'s
`run_mds()`), with binary observed/unobserved weights -- missing pairs are simply excluded from the
fit, with no imputation. One fit per dimensionality 2..`MAX_DIMS` per cohort
(28 fits total); each real-scale fit takes ~15-30s, so results are cached to
`analysis/results/behavioral/mds_cache/` (one `.npz` per cohort x dimension) and re-runs of this
notebook reuse the cache instead of refitting.

In [3]:
MAX_DIMS = 15
MDS_DIMS = list(range(2, MAX_DIMS + 1))

mds_results = {
    c: run_mds_sweep(rdm[c], MDS_DIMS, c, MDS_CACHE_DIR, max_iters=1000, convergence_tol=1e-6)
    for c in COHORTS
}
for c in COHORTS:
    print(f"  {COHORT_LABEL[c]}: {len(mds_results[c])} dimensionalities ready (cached under {MDS_CACHE_DIR})")

  Pre-SHINE: 14 dimensionalities ready (cached under C:\Users\nirjo\Documents\University\PhD\Projects\hierarchical_image_database\.claude\worktrees\awesome-leakey-d0b3a3\analysis\results\behavioral\mds_cache)
  Post-SHINE: 14 dimensionalities ready (cached under C:\Users\nirjo\Documents\University\PhD\Projects\hierarchical_image_database\.claude\worktrees\awesome-leakey-d0b3a3\analysis\results\behavioral\mds_cache)


### 1.2 Stress scree plot

Weighted-SMACOF stress-1 across dimensionality, one trace per cohort -- the validated
(no-imputation) dimensionality diagnostic.

In [4]:
fig_stress_scree(mds_results, MDS_DIMS).show()

### 1.3 Eigenvalue spectrum of the MDS embeddings

Each SMACOF embedding is a configuration of 725 points in `d` dimensions. Its eigenvalue spectrum is the
variance along each of the `d` principal axes of that configuration (equivalently, the eigenvalues of the
double-centred Gram matrix of the embedding's own pairwise distances), here as a proportion of the total
variance so that the pre and post fits, whose overall scales are arbitrary, are comparable. Nothing is
imputed: the fit ignores unobserved pairs, so the spectrum describes the fitted geometry itself. Three
dimensionalities are shown (3, 9, 15), sharing the y-axis. A spectrum that falls steeply means a few axes
carry most of the variance; a gap between the cohorts' curves means they spread variance across the axes
differently.

In [5]:
SPECTRUM_DIMS = (3, 9, 15)
spectra = {d: {c: embedding_eigenspectrum(mds_results[c][d]["conf"]) for c in COHORTS} for d in SPECTRUM_DIMS}
fig_embedding_spectrum(spectra).show()

### 1.4 Pre- vs. post-SHINE structural agreement

For the raw RDMs and for each embedding dimensionality, do the pre- and post-SHINE cohorts agree
on the *same* similarity structure? Kendall tau and Spearman rho are computed on the raw RDMs too
(restricted to jointly-covered pairs), but Procrustes m^2 needs a complete point configuration, so
it's only defined once an MDS embedding exists (`NaN` for "raw RDM" below). Procrustes m^2 is
shown as `1 - m^2` so all three traces read "bigger = better".

In [6]:
agreement: dict[str, dict] = {}
agreement["raw RDM"] = {**masked_rank_correlation(rdm["pre"], rdm["post"]), "procrustes_m2": None}
for d in MDS_DIMS:
    ks = masked_rank_correlation(mds_results["pre"][d]["confdist"], mds_results["post"][d]["confdist"])
    m2 = procrustes_m2(mds_results["pre"][d]["conf"], mds_results["post"][d]["conf"])
    agreement[str(d)] = {**ks, "procrustes_m2": m2}

agreement_table = pd.DataFrame(agreement).T[["kendall", "spearman", "procrustes_m2", "n_overlap"]]
agreement_table

,kendall,spearman,procrustes_m2,n_overlap
raw RDM,0.147864,0.219826,NaN,107040.0
2,0.502793,0.689974,0.305794,262450.0
3,0.435604,0.613894,0.375806,262450.0
4,0.404671,0.573814,0.442914,262450.0
5,0.406591,0.574581,0.477917,262450.0
6,0.402729,0.567673,0.550900,262450.0
7,0.418474,0.586165,0.509305,262450.0
8,0.429431,0.599686,0.527540,262450.0
9,0.436127,0.606954,0.534334,262450.0
10,0.447335,0.620495,0.535048,262450.0


In [7]:
fig_cross_cohort_agreement(agreement, MDS_DIMS).show()

## 2. Cohort-label permutation test of pre-post agreement

Section 1.4 reports how well the two cohorts' RDMs agree (raw RDM tau 0.15, rho 0.22). With over 100,000
jointly rated pairs any positive correlation is trivially significant, so the informative question is
whether the two cohorts agree *less than exchangeable cohorts would*, i.e. whether SHINE changes the
similarity structure beyond sampling noise. The test, per statistic (Kendall tau, Spearman rho on
jointly covered pairs):

1. Observed: build the pre RDM and post RDM ("simple" aggregation) and correlate them.
2. Pool all retained subjects and shuffle the cohort label at the subject level (group sizes kept).
3. Rebuild both RDMs from the shuffled groups and correlate them. Repeat `N_PERM` times.
4. One-sided p = (1 + number of shuffled values <= observed) / (1 + `N_PERM`).

The null already carries the same noise level, sample size and pair coverage, so its mean is the
agreement expected if the cohorts were identical. This tests for a difference, not for equivalence: a
non-small p means no detectable difference, not proof of none. It assumes subjects are exchangeable
across cohorts apart from SHINE. The null draws are cached under
`analysis/results/behavioral/permutation_cache/`.

In [8]:
N_PERM = 500
N_JOBS = 6

cohorts = load_retained_cohorts(cache_dir)
subject_ids = {c: cohorts["participant_ids"][c].tolist() for c in COHORTS}
trials_by_subject, n_conds = parse_subject_trials(cohorts["df_trials"], subject_ids)
rdm_builder = partial(rdm_from_subject_trials, trials_by_subject=trials_by_subject, n_conds=n_conds)

permutation = permutation_test_cohort_agreement(
    rdm_builder, subject_ids["pre"], subject_ids["post"], n_perm=N_PERM, seed=0, n_jobs=N_JOBS,
    cache_path=PERMUTATION_CACHE_DIR / f"cohort_agreement_n{N_PERM}_seed0.npz",
)
pd.DataFrame({key: permutation[key] for key in ("kendall", "spearman")}).T.round(4)

,observed,null_mean,null_low,null_high,p
kendall,0.1479,0.1481,0.1451,0.1511,0.4431
spearman,0.2198,0.2202,0.2160,0.2245,0.4351


In [9]:
fig_permutation_null(permutation, "kendall", statistic_label="Kendall tau").show()
fig_permutation_null(permutation, "spearman", statistic_label="Spearman rho").show()

## 3. Equivalence of pre-post agreement

Section 2 tests whether the cohorts agree *less* than exchangeable cohorts would; a non-small p there is
not evidence that they agree as much. This section tests equivalence directly. The effect is

    D = (observed agreement) - (mean of the label-permutation null)

for Kendall tau and Spearman rho, where "agreement" is the rank correlation between the pre and post RDMs on
jointly covered pairs, and the null mean is the agreement expected if the cohorts were identical (section 2).
D = 0 means the cohorts agree exactly as much as two exchangeable groups would. The question is whether D is
within a small bound of 0.

**Where the 90% CI of D comes from: a delete-a-group jackknife.** The CI needs the sampling uncertainty of D,
i.e. how much D would change with a different sample of subjects. The spread of the permutation null cannot
provide it: every shuffle reuses the same subjects, so it understates this uncertainty. The jackknife measures
it by recomputing D on many slightly smaller samples, each leaving out a different slice of subjects, and
looking at how much D moves. "Delete-one" would drop one subject at a time (155 recomputations, each with its
own null); "delete-a-group" drops a whole group at a time, so only G = 20 recomputations are needed.

```
D_obs = agreement(pre, post) - mean(null)               # null: the 500 label shuffles of section 2

split the pre subjects into G = 20 random groups          # about 4 subjects per group
split the post subjects into G = 20 random groups

for g in 1..G:
    pre_g  = all pre subjects except pre group g          # about 95% of the subjects remain
    post_g = all post subjects except post group g
    null_g = mean of agreement(A, B) over 10 shuffles of the pooled subjects of (pre_g, post_g),
             each shuffle re-splitting them into groups of the same sizes as pre_g and post_g
    D_g    = agreement(pre_g, post_g) - null_g

SE     = sqrt( (G - 1) / G  *  sum over g of (D_g - mean(D_1..D_G))^2 )
90% CI = D_obs  +/-  1.645 * SE
```

The factor (G - 1) / G corrects for the replicates being highly overlapping (each keeps about 95% of the
subjects), which makes them vary less than independent samples would. Only 10 shuffles per leave-out are used
for `null_g`; this adds a little noise to each D_g, which the SE absorbs.

**The bound.** As for the other equivalence tests, it is the smallest bound that would show equivalence with
80% power if D were truly 0: (1.645 + 0.84) x SE, rounded up to 0.01 in tau/rho units. D is judged equivalent
when the whole 90% CI lies inside the bound.

In [10]:
equivalence_agreement = equivalence_test_cohort_agreement(
    rdm_builder, subject_ids["pre"], subject_ids["post"], permutation,
    n_groups=20, n_null=10, seed=0, n_jobs=N_JOBS,
    cache_path=PERMUTATION_CACHE_DIR / "cohort_agreement_jackknife_g20_n10_seed0.npz",
)
for statistic, result in equivalence_agreement.items():
    print(f"{statistic}: jackknife SE={result['se']:.4f}, 80%-power sensitivity={result['sensitivity']:.4f}")
    print_equivalence_line(result)
pd.DataFrame(equivalence_agreement).T[["effect", "ci_low", "ci_high", "se", "sensitivity", "bound", "equivalent"]]

kendall: jackknife SE=0.0028, 80%-power sensitivity=0.0071
  observed - exchangeable expectation (kendall): effect=-0.000  90% CI [-0.005, 0.004]  bound=+/-0.01 (equivalent)
spearman: jackknife SE=0.0042, 80%-power sensitivity=0.0104
  observed - exchangeable expectation (spearman): effect=-0.000  90% CI [-0.007, 0.006]  bound=+/-0.02 (equivalent)


,effect,ci_low,ci_high,se,sensitivity,bound,equivalent
kendall,-0.0002,-0.0049,0.0045,0.00284,0.00707,0.01,True
spearman,-0.0004,-0.0072,0.0065,0.00417,0.01037,0.02,True


In [11]:
fig_equivalence_forest(
    {"Kendall tau": equivalence_agreement["kendall"], "Spearman rho": equivalence_agreement["spearman"]},
    x_title="Observed minus exchangeable expectation, 90% CI", title="Equivalence: pre-post RDM agreement",
).show()